In [3]:
# =====================================================================
# [Step 0] 라이브러리 준비 (딥러닝 영끌 모드!)
# =====================================================================
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import copy # 🌟 치트키 추가: 모델의 '리즈 시절 뇌'를 복사해두는 도구!
import random # 🌟 컨디션 고정용 도구 추가!
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

print("🚀 딥러닝 끝판왕 모드 가동! 영혼까지 끌어모으긔! 삐리릭-")

# 🌟 잼띠의 특별 부적: AI 컨디션(랜덤 운빨) 고정하기! 🌟
# 이걸 해두면 언제 돌려도 똑같은 점수가 나오게 최대한 억제해 주긔!
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42) # 부적 발동!

# =====================================================================
# [Step 1] 데이터 준비 및 전처리
# =====================================================================
df = pd.read_csv('steam_games_preprocessed.csv')

target_col = "main_genre"
X = df.drop(columns=[target_col])
y = df[target_col]

X = X.astype(float).fillna(0)

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
num_classes = len(label_encoder.classes_)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
# 🌟 배치 사이즈를 128 -> 64로 줄임! (한 번에 조금씩 더 꼼꼼히 문제 풀기!)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

# =====================================================================
# [Step 2] 🌟 딥러닝 치트키 1: 클래스 가중치 삭제 (점수 떡락 방지!) 🌟
# =====================================================================
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# =====================================================================
# [Step 3] 🌟 딥러닝 치트키 2: 스킵 커넥션 (Skip Connection) 아키텍처 🌟
# =====================================================================
class AdvancedSteamNet(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(AdvancedSteamNet, self).__init__()

        self.layer1 = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.BatchNorm1d(512),
            nn.GELU(),
            nn.Dropout(0.3)
        )

        self.layer2 = nn.Sequential(
            nn.Linear(512, 512),
            nn.BatchNorm1d(512),
            nn.GELU(),
            nn.Dropout(0.3)
        )

        self.layer3 = nn.Sequential(
            nn.Linear(512, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(0.2)
        )

        self.out = nn.Linear(128, output_dim)

    def forward(self, x):
        x1 = self.layer1(x)
        x2 = self.layer2(x1)
        x_skip = x1 + x2
        x3 = self.layer3(x_skip)
        return self.out(x3)

input_dim = X_train.shape[1]
model = AdvancedSteamNet(input_dim, num_classes).to(device)

# =====================================================================
# [Step 4] 🌟 딥러닝 치트키 3: AdamW & 스케줄러 & 얼리스탑 🌟
# =====================================================================
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=0.005, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

epochs = 60
print(f"🔥 영끌 딥러닝 학습 시작! (총 {epochs} Epochs)")

best_acc = 0.0
patience_counter = 0
patience_limit = 7
best_model_wts = copy.deepcopy(model.state_dict()) # 🌟 리즈 시절 뇌(가중치)를 저장할 변수!

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    epoch_loss = running_loss / len(train_loader)
    scheduler.step(epoch_loss)

    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        inputs, labels = X_test_tensor.to(device), y_test_tensor.to(device)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        total = labels.size(0)
        correct = (predicted == labels).sum().item()

    val_acc = (correct / total) * 100

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"[Epoch {epoch+1:02d}/{epochs}] 훈련 로스: {epoch_loss:.4f} | 📝 모의고사 점수(Val Acc): {val_acc:.2f}%")

    if val_acc > best_acc:
        best_acc = val_acc
        patience_counter = 0
        best_model_wts = copy.deepcopy(model.state_dict()) # 🌟 점수 갱신할 때마다 그 순간의 뇌를 복사해둠!
    else:
        patience_counter += 1

    if patience_counter >= patience_limit:
        print("\n🛑 [얼리스탑 발동!] AI 뇌절 오기 전에 멱살 잡고 강제 종료시켰긔!")
        print(f"🛑 가장 똑똑했을 때의 점수: {best_acc:.2f}%")
        break

# 🌟 학습 종료 후, 가장 똑똑했던 리즈 시절의 뇌로 다시 갈아끼우기!
model.load_state_dict(best_model_wts)

# =====================================================================
# [Step 5] 최종 시식 (정확도 평가)
# =====================================================================
model.eval()
correct = 0
total = 0

with torch.no_grad():
    inputs, labels = X_test_tensor.to(device), y_test_tensor.to(device)
    outputs = model(inputs)
    _, predicted = torch.max(outputs.data, 1)

    total = labels.size(0)
    correct = (predicted == labels).sum().item()

accuracy = (correct / total) * 100
print("==================================================")
print(f"💖 스팀 딥러닝 끝판왕 최종 정확도: {accuracy:.2f}% 💖")
print("==================================================")

🚀 딥러닝 끝판왕 모드 가동! 영혼까지 끌어모으긔! 삐리릭-
🔥 영끌 딥러닝 학습 시작! (총 60 Epochs)
[Epoch 01/60] 훈련 로스: 1.4647 | 📝 모의고사 점수(Val Acc): 42.98%
[Epoch 05/60] 훈련 로스: 1.3990 | 📝 모의고사 점수(Val Acc): 45.53%
[Epoch 10/60] 훈련 로스: 1.3843 | 📝 모의고사 점수(Val Acc): 45.71%
[Epoch 15/60] 훈련 로스: 1.3725 | 📝 모의고사 점수(Val Acc): 42.53%
[Epoch 20/60] 훈련 로스: 1.3606 | 📝 모의고사 점수(Val Acc): 45.65%

🛑 [얼리스탑 발동!] AI 뇌절 오기 전에 멱살 잡고 강제 종료시켰긔!
🛑 가장 똑똑했을 때의 점수: 45.82%
💖 스팀 딥러닝 끝판왕 최종 정확도: 45.82% 💖
